In [1]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import time
from collections import defaultdict
import random

import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:

def set_seed(seed=456):
    """Set all random seeds for reproducibility"""
    #Python random
    random.seed(seed)

    # NumPy random
    np.random.seed(seed)

    # PyTorch random
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # For deterministic behavior (slower but reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # For DataLoader workers
    os.environ['PYTHONHASHSEED'] = str(seed)

    print(f"🌱 All random seeds set to: {seed}")
    print("✅ Training will now be reproducible!")

# รันทันที!
set_seed(456)

🌱 All random seeds set to: 456
✅ Training will now be reproducible!


In [3]:
# STEP 0 — mount (ทำครั้งเดียวต่อ session)
from google.colab import drive
drive.mount('/content/drive')

# STEP 1 — ตรวจว่าพาธถูกจริงไหม
!ls "/content/drive/MyDrive/Project Deeplearning/TACO" | head

# STEP 2 — ถ้ารายการโผล่ถูก ให้รันโค้ดต่อได้เลย
from pathlib import Path, PurePosixPath
ANN_FILE = Path("/content/drive/MyDrive/Project Deeplearning/TACO/train_annotations.json")
print("exists?", ANN_FILE.exists())        # ต้องขึ้น True

with open(ANN_FILE, "r") as f:
    taco_json = json.load(f)
category_map = {c["id"]: c["name"] for c in taco_json["categories"]}
print("✓ loaded", len(category_map), "classes")


Mounted at /content/drive
annotations.json
batch_1
batch_10
batch_11
batch_12
batch_13
batch_14
batch_15
batch_2
batch_3
exists? True
✓ loaded 60 classes


In [4]:
# สมมติเราเซ็ต ROOT ไว้แล้วในเซลล์ถัดไปว่า
ROOT = Path("/content/drive/MyDrive/Project Deeplearning/TACO")

from pathlib import Path

ANN_FILE = Path("/content/drive/MyDrive/Project Deeplearning/TACO/train_annotations.json")
# หรือจะใช้ SPLITS["train"] ภายหลังก็ได้

with open(ANN_FILE) as f:
    taco_json = json.load(f)

category_map = {cat['id']: cat['name'] for cat in taco_json['categories']}
print(f"✓ category_map loaded — {len(category_map)} classes")


✓ category_map loaded — 60 classes


In [5]:
from torch.utils.data import Dataset
class TacoDataset(Dataset):
    def __init__(self, root, annotations: dict, transforms=None):
        self.root       = root                      # Path/str ของโฟลเดอร์รูป
        self.transforms = transforms
        self.annos      = annotations["annotations"]
        self.images     = annotations["images"]
        self.id2img     = {img["id"]: img for img in self.images}

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.root, img_info["file_name"])
        image    = Image.open(img_path).convert("RGB")

        # ----- annotations ของภาพนี้ -----
        annos = [a for a in self.annos if a["image_id"] == img_info["id"]]

        boxes, labels = [], []
        for a in annos:
            x, y, w, h = a["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(a["category_id"])

        boxes  = torch.as_tensor(boxes,  dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            "boxes":    boxes,
            "labels":   labels,
            "image_id": torch.tensor([img_info["id"]]),   # ใช้ id จริง
            "area":     (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            "iscrowd":  torch.zeros((len(boxes),), dtype=torch.int64),
        }

        # ---- transforms ----
        if self.transforms:
            image = self.transforms(image)
        # ถ้า transforms ไม่ได้แปลงเป็น tensor ให้ทำ
        if not isinstance(image, torch.Tensor):
            image = F.to_tensor(image)

        return image, target



In [6]:

from torchvision.transforms import functional as F
from torch.utils.data import DataLoader

ROOT = "/content/drive/MyDrive/Project Deeplearning/TACO"

# --- train ---
with open(f"{ROOT}/train_annotations.json") as f:
    train_json = json.load(f)

train_dataset = TacoDataset(
    root=f"{ROOT}/train",
    annotations=train_json,
    transforms=lambda img: F.to_tensor(img),
)

# --- val ---
with open(f"{ROOT}/val_annotations.json") as f:
    val_json = json.load(f)

val_dataset = TacoDataset(
    root=f"{ROOT}/val",
    annotations=val_json,
    transforms=lambda img: F.to_tensor(img),
)

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=4,
                          shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=2,
                          shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f"train: {len(train_dataset)} • val: {len(val_dataset)}")



train: 1050 • val: 225


In [7]:
"""# เพิ่ม cell ใหม่หลังจาก cell ที่สร้าง TacoDataset
import json                           # ← ถ้ายังไม่ใส่
import torchvision.transforms as T
import random

class EnhancedTransforms:
    def __init__(self, train=True):
        self.train = train

    def __call__(self, image):
        if self.train:
            # Random horizontal flip
            if random.random() < 0.5:
                image = F.hflip(image)

            # Color jittering (เล็กน้อยเพื่อไม่ให้ดูไม่เป็นธรรมชาติ)
            if random.random() < 0.3:
                image = T.ColorJitter(
                    brightness=0.15,
                    contrast=0.15,
                    saturation=0.15
                )(image)

        # Convert to tensor
        return F.to_tensor(image)

# สร้าง dataset ใหม่ที่มี augmentation ดีขึ้น
train_dataset_enhanced = TacoDataset(
    root=f"{ROOT}/train",
    annotations=train_json,
    transforms=EnhancedTransforms(train=True)
)

val_dataset_enhanced = TacoDataset(
    root=f"{ROOT}/val",
    annotations=val_json,
    transforms=EnhancedTransforms(train=False)
)"""



'# เพิ่ม cell ใหม่หลังจาก cell ที่สร้าง TacoDataset\nimport json                           # ← ถ้ายังไม่ใส่\nimport torchvision.transforms as T\nimport random\n\nclass EnhancedTransforms:\n    def __init__(self, train=True):\n        self.train = train\n\n    def __call__(self, image):\n        if self.train:\n            # Random horizontal flip\n            if random.random() < 0.5:\n                image = F.hflip(image)\n\n            # Color jittering (เล็กน้อยเพื่อไม่ให้ดูไม่เป็นธรรมชาติ)\n            if random.random() < 0.3:\n                image = T.ColorJitter(\n                    brightness=0.15,\n                    contrast=0.15,\n                    saturation=0.15\n                )(image)\n\n        # Convert to tensor\n        return F.to_tensor(image)\n\n# สร้าง dataset ใหม่ที่มี augmentation ดีขึ้น\ntrain_dataset_enhanced = TacoDataset(\n    root=f"{ROOT}/train",\n    annotations=train_json,\n    transforms=EnhancedTransforms(train=True)\n)\n\nval_dataset_enhanced 

In [8]:
import torchvision.transforms as T

class MultiScaleEnhancedTransforms:
    def __init__(self, train=True):
        self.train = train
        self.scales = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]  # Scale factors
        self.input_sizes = [480, 512, 544, 576, 608]  # Different input sizes

    def __call__(self, image):
        if self.train:
            # 1. Multi-scale resizing
            scale = random.choice(self.scales)
            w, h = image.size
            new_w, new_h = int(w * scale), int(h * scale)

            # Resize image
            image = image.resize((new_w, new_h), Image.BILINEAR)

            # 2. Random input size
            target_size = random.choice(self.input_sizes)

            # 3. Handle different scales
            if scale > 1.0:
                # Random crop for upscaled images
                image = T.RandomCrop(target_size, padding=20)(image)
            else:
                # Pad and crop for downscaled images
                pad_size = max(0, target_size - min(new_w, new_h))
                image = T.Pad(pad_size//2, fill=0)(image)
                image = T.CenterCrop(target_size)(image)

            # Ensure final size
            image = T.Resize((target_size, target_size))(image)

            # 4. Additional augmentations
            if random.random() < 0.5:
                image = T.functional.hflip(image)

            if random.random() < 0.3:
                # Random rotation
                angle = random.uniform(-10, 10)
                image = T.functional.rotate(image, angle, expand=False)

            if random.random() < 0.4:
                # Color augmentation
                image = T.ColorJitter(
                    brightness=0.2,
                    contrast=0.2,
                    saturation=0.2,
                    hue=0.05
                )(image)

        return F.to_tensor(image)


# สร้าง dataset ใหม่ที่มี augmentation ดีขึ้น

train_dataset_enhanced = TacoDataset(
    root=f"{ROOT}/train",
    annotations=train_json,
    transforms=MultiScaleEnhancedTransforms(train=True)  # ← ใหม่
)

val_dataset_enhanced = TacoDataset(
    root=f"{ROOT}/val",
    annotations=val_json,
    transforms=MultiScaleEnhancedTransforms(train=False)  # ← ใหม่
)

In [9]:
# =============================================================================
# CELL ใหม่ที่ 1: วิเคราะห์ Class Distribution
# =============================================================================
from collections import Counter
from torch.utils.data import WeightedRandomSampler
import matplotlib.pyplot as plt

def analyze_class_distribution(annotations, category_map):
    """วิเคราะห์การกระจายของคลาสต่างๆ ใน dataset"""

    # นับจำนวน annotation ของแต่ละคลาส
    class_counts = Counter([ann['category_id'] for ann in annotations])

    print("🔍 TACO Dataset Class Distribution Analysis")
    print("=" * 60)

    # สถิติพื้นฐาน
    total_annotations = sum(class_counts.values())
    num_classes = len(class_counts)
    avg_per_class = total_annotations / num_classes

    print(f"📊 Dataset Statistics:")
    print(f"   Total annotations: {total_annotations:,}")
    print(f"   Number of classes: {num_classes}")
    print(f"   Average per class: {avg_per_class:.1f}")

    # เรียงลำดับจากมากไปน้อย
    sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)

    # แสดง Top 10 คลาสที่มีมากที่สุด
    print(f"\n🔝 TOP 10 Most Frequent Classes:")
    print("-" * 50)
    for i, (cat_id, count) in enumerate(sorted_classes[:10]):
        class_name = category_map.get(cat_id, f"Unknown_{cat_id}")
        percentage = (count / total_annotations) * 100
        print(f"{i+1:2d}. {class_name:<25} {count:5d} samples ({percentage:5.1f}%)")

    # แสดง Bottom 10 คลาสที่มีน้อยที่สุด
    print(f"\n🔻 BOTTOM 10 Least Frequent Classes:")
    print("-" * 50)
    for i, (cat_id, count) in enumerate(sorted_classes[-10:]):
        class_name = category_map.get(cat_id, f"Unknown_{cat_id}")
        percentage = (count / total_annotations) * 100
        rank = len(sorted_classes) - 10 + i + 1
        print(f"{rank:2d}. {class_name:<25} {count:5d} samples ({percentage:5.1f}%)")

    # คำนวณ Imbalance Ratio
    max_count = sorted_classes[0][1]
    min_count = sorted_classes[-1][1]
    imbalance_ratio = max_count / min_count

    print(f"\n⚖️ Class Imbalance Analysis:")
    print(f"   Most frequent class: {max_count:,} samples")
    print(f"   Least frequent class: {min_count:,} samples")
    print(f"   Imbalance ratio: {imbalance_ratio:.1f}:1")

    if imbalance_ratio > 10:
        print("   ⚠️  SEVERE IMBALANCE detected! (>10:1)")
    elif imbalance_ratio > 5:
        print("   ⚠️  MODERATE IMBALANCE detected (>5:1)")
    else:
        print("   ✅ Relatively balanced dataset")

    return class_counts, sorted_classes

# เรียกใช้ฟังก์ชันวิเคราะห์
print("🚀 Analyzing TACO dataset class distribution...")
class_distribution, sorted_classes = analyze_class_distribution(
    train_json['annotations'],
    category_map
)

🚀 Analyzing TACO dataset class distribution...
🔍 TACO Dataset Class Distribution Analysis
📊 Dataset Statistics:
   Total annotations: 3,215
   Number of classes: 58
   Average per class: 55.4

🔝 TOP 10 Most Frequent Classes:
--------------------------------------------------
 1. Cigarette                   413 samples ( 12.8%)
 2. Plastic film                313 samples (  9.7%)
 3. Unlabeled litter            297 samples (  9.2%)
 4. Clear plastic bottle        202 samples (  6.3%)
 5. Other plastic               170 samples (  5.3%)
 6. Other plastic wrapper       157 samples (  4.9%)
 7. Drink can                   154 samples (  4.8%)
 8. Plastic bottle cap          133 samples (  4.1%)
 9. Broken glass                132 samples (  4.1%)
10. Plastic straw               122 samples (  3.8%)

🔻 BOTTOM 10 Least Frequent Classes:
--------------------------------------------------
49. Shoe                          5 samples (  0.2%)
50. Tupperware                    4 samples (  0.1%)


In [10]:
# =============================================================================
# CELL ใหม่ที่ 2: สร้าง Weighted Sampler Function
# =============================================================================

def create_weighted_sampler(dataset, annotations, category_map, method='inverse_freq'):
    """
    สร้าง WeightedRandomSampler เพื่อแก้ปัญหา class imbalance

    Args:
        dataset: TacoDataset object
        annotations: list ของ annotations
        category_map: mapping จาก category_id ไป class_name
        method: วิธีการคำนวณ weight ('inverse_freq' หรือ 'sqrt_inverse_freq')
    """

    print(f"🎯 Creating Weighted Sampler (method: {method})")
    print("-" * 50)

    # Step 1: นับจำนวน instances ของแต่ละคลาส
    class_counts = Counter([ann['category_id'] for ann in annotations])
    total_annotations = len(annotations)
    num_classes = len(class_counts)

    print(f"   Total annotations: {total_annotations:,}")
    print(f"   Number of classes: {num_classes}")

    # Step 2: คำนวณ weight สำหรับแต่ละคลาส
    class_weights = {}

    for class_id, count in class_counts.items():
        if method == 'inverse_freq':
            # Inverse frequency weighting (ยิ่งหายาก weight ยิ่งสูง)
            weight = total_annotations / (num_classes * count)
        elif method == 'sqrt_inverse_freq':
            # Square root inverse frequency (เบาลงหน่อย)
            weight = (total_annotations / (num_classes * count)) ** 0.5
        else:
            weight = 1.0

        class_weights[class_id] = weight

    # Step 3: สร้าง sample weights สำหรับแต่ละภาพ
    sample_weights = []
    images_with_weights = []

    for idx, img_info in enumerate(dataset.images):
        img_id = img_info['id']

        # หา annotations ทั้งหมดในภาพนี้
        img_annotations = [ann for ann in annotations if ann['image_id'] == img_id]

        if img_annotations:
            # คำนวณ weight เฉลี่ยของ objects ทั้งหมดในภาพ
            weights_in_image = [class_weights[ann['category_id']] for ann in img_annotations]
            avg_weight = sum(weights_in_image) / len(weights_in_image)
        else:
            # ภาพที่ไม่มี annotation ให้ weight ต่ำ
            avg_weight = 0.1

        sample_weights.append(avg_weight)
        images_with_weights.append((img_info['file_name'], avg_weight, len(img_annotations)))

    # Step 4: แสดงสถิติ
    min_weight = min(sample_weights)
    max_weight = max(sample_weights)
    avg_weight = sum(sample_weights) / len(sample_weights)

    print(f"   Sample weights range: {min_weight:.3f} - {max_weight:.3f}")
    print(f"   Average sample weight: {avg_weight:.3f}")
    print(f"   Weight ratio: {max_weight/min_weight:.1f}:1")

    # แสดงตัวอย่างภาพที่มี weight สูงสุด/ต่ำสุด
    images_with_weights.sort(key=lambda x: x[1], reverse=True)

    print(f"\n📈 Top 5 Highest Weight Images:")
    for i, (filename, weight, num_objects) in enumerate(images_with_weights[:5]):
        print(f"   {i+1}. {filename:<30} weight: {weight:.3f} ({num_objects} objects)")

    print(f"\n📉 Top 5 Lowest Weight Images:")
    for i, (filename, weight, num_objects) in enumerate(images_with_weights[-5:]):
        print(f"   {i+1}. {filename:<30} weight: {weight:.3f} ({num_objects} objects)")

    # Step 5: สร้าง WeightedRandomSampler
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True  # อนุญาตให้เลือกภาพเดิมได้มากกว่า 1 ครั้งใน 1 epoch
    )

    print(f"\n✅ WeightedRandomSampler created successfully!")
    print(f"   Total samples per epoch: {len(sample_weights)}")
    print(f"   Replacement: True (same image can appear multiple times)")

    return sampler, class_weights

# สร้าง weighted sampler
print("🎯 Creating weighted sampler for training set...")
weighted_sampler, class_weights = create_weighted_sampler(
    train_dataset_enhanced,  # ใช้ dataset ที่มี augmentation
    train_json['annotations'],
    category_map,
    method='inverse_freq'  # ลองเปลี่ยนเป็น 'sqrt_inverse_freq' ถ้าต้องการ weight เบาลง
)

🎯 Creating weighted sampler for training set...
🎯 Creating Weighted Sampler (method: inverse_freq)
--------------------------------------------------
   Total annotations: 3,215
   Number of classes: 58
   Sample weights range: 0.134 - 27.716
   Average sample weight: 1.191
   Weight ratio: 206.5:1

📈 Top 5 Highest Weight Images:
   1. batch_8/000082.jpg             weight: 27.716 (1 objects)
   2. batch_4/000026.JPG             weight: 27.716 (1 objects)
   3. batch_8/000062.jpg             weight: 27.716 (1 objects)
   4. batch_8/000070.jpg             weight: 18.477 (1 objects)
   5. batch_1/000119.JPG             weight: 18.477 (1 objects)

📉 Top 5 Lowest Weight Images:
   1. batch_12/000024.jpg            weight: 0.142 (25 objects)
   2. batch_10/000020.jpg            weight: 0.138 (11 objects)
   3. batch_10/000064.jpg            weight: 0.134 (1 objects)
   4. batch_14/000011.jpg            weight: 0.134 (1 objects)
   5. batch_12/000003.jpg            weight: 0.134 (1 objects)


In [11]:
# DataLoader ที่ปรับปรุง
train_loader_enhanced = DataLoader(
    train_dataset_enhanced, batch_size=2, sampler=weighted_sampler, #shuffle=True
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)

val_loader_enhanced = DataLoader(
    val_dataset_enhanced, batch_size=1, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True
)

print("✅ Enhanced data augmentation ready!")
print(f"   Train: {len(train_dataset_enhanced)} samples")
print(f"   Val: {len(val_dataset_enhanced)} samples")

✅ Enhanced data augmentation ready!
   Train: 1050 samples
   Val: 225 samples


In [12]:
import torch, torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_model(num_classes: int):
    # ── 1) load pretrained Faster-R-CNN-V2 ────────────────────────────
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
    model   = fasterrcnn_resnet50_fpn_v2(weights=weights)

    # ── 2) replace the classifier head ────────────────────────────────
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)

    # ── 3) partial fine-tuning  (unfreeze layer4 ของ backbone) ────────
    for p in model.backbone.parameters():
        p.requires_grad_(False)

    for name, p in model.backbone.body.named_parameters():
        if "layer4" in name:
            p.requires_grad_(True)

    return model.to(device)

# -------- create model --------
num_classes = len(category_map) + 1     # +1 background
model = get_model(num_classes)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
100%|██████████| 167M/167M [00:00<00:00, 234MB/s]


In [13]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ── เตรียม COCO ground-truth ของ val set (ทำครั้งเดียว) ──
coco_gt = COCO(f"{ROOT}/val_annotations.json")   # ← path val json

# เพิ่ม cell ใหม่หลังจาก evaluate_map function เดิม
def enhanced_evaluate_map(model, data_loader, device, coco_gt, category_map, conf_threshold=0.5):
    """Enhanced evaluation with detailed metrics"""
    model.eval()
    results = []
    total_predictions = 0

    print(f"🔍 Running evaluation with confidence threshold: {conf_threshold}")

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Evaluating", leave=False):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for tgt, out in zip(targets, outputs):
                img_id = int(tgt["image_id"].item())

                boxes = out["boxes"].cpu()
                scores = out["scores"].cpu()
                labels = out["labels"].cpu()

                # Filter by confidence
                keep = scores > conf_threshold
                boxes = boxes[keep]
                scores = scores[keep]
                labels = labels[keep]

                total_predictions += len(boxes)

                # Convert xyxy to xywh
                if len(boxes) > 0:
                    boxes[:, 2:] -= boxes[:, :2]

                    for box, score, label in zip(boxes, scores, labels):
                        results.append({
                            "image_id": img_id,
                            "category_id": int(label.item()),
                            "bbox": [round(x, 2) for x in box.tolist()],
                            "score": float(score.item()),
                        })

    print(f"   Total predictions: {total_predictions}")

    if not results:
        print("⚠️ No predictions above confidence threshold")
        return 0.0, 0.0, {}

    # COCO evaluation
    try:
        coco_gt.dataset.setdefault("info", {})
        coco_dt = coco_gt.loadRes(results)
        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

        map_50_95 = coco_eval.stats[0]  # mAP@0.5:0.95
        map_50 = coco_eval.stats[1]     # mAP@0.5

        return map_50_95, map_50, results

    except Exception as e:
        print(f"⚠️ Evaluation error: {e}")
        return 0.0, 0.0, {}

# ฟังก์ชันวิเคราะห์ per-class
def analyze_class_performance(results, category_map):
    """Analyze per-class detection performance"""
    if not results:
        return {}

    class_counts = {}
    for result in results:
        cat_id = result['category_id']
        cat_name = category_map.get(cat_id, f"Unknown_{cat_id}")
        class_counts[cat_name] = class_counts.get(cat_name, 0) + 1

    # Sort by count
    sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)

    print("\n📊 Detection Count by Class (Top 15):")
    print("-" * 45)
    for i, (class_name, count) in enumerate(sorted_classes[:15]):
        print(f"{i+1:2d}. {class_name:<25} {count:4d} detections")

    return class_counts

print("✅ Enhanced evaluation functions ready!")

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
✅ Enhanced evaluation functions ready!


In [14]:
#เพิ่มฟังก์ชัน Size-Aware Evaluation

def size_aware_evaluate_map(model, data_loader, device, coco_gt, category_map):
    """Enhanced evaluation with size-aware confidence thresholding"""
    model.eval()
    results = []
    total_predictions = 0

    # Define size-specific thresholds
    SMALL_THRESHOLD = 0.3   # Lower for small objects
    MEDIUM_THRESHOLD = 0.4  # Medium for medium objects
    LARGE_THRESHOLD = 0.5   # Keep high for large objects

    print(f"🔍 Running size-aware evaluation")
    print(f"   Small objects threshold: {SMALL_THRESHOLD}")
    print(f"   Medium objects threshold: {MEDIUM_THRESHOLD}")
    print(f"   Large objects threshold: {LARGE_THRESHOLD}")

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Size-aware Evaluation", leave=False):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for tgt, out in zip(targets, outputs):
                img_id = int(tgt["image_id"].item())

                boxes = out["boxes"].cpu()
                scores = out["scores"].cpu()
                labels = out["labels"].cpu()

                # Calculate areas for size classification
                areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

                # Apply size-specific thresholds
                keep_indices = []
                small_count, medium_count, large_count = 0, 0, 0

                for i, (area, score) in enumerate(zip(areas, scores)):
                    if area < 32*32:  # Small objects
                        if score > SMALL_THRESHOLD:
                            keep_indices.append(i)
                            small_count += 1
                    elif area < 96*96:  # Medium objects
                        if score > MEDIUM_THRESHOLD:
                            keep_indices.append(i)
                            medium_count += 1
                    else:  # Large objects
                        if score > LARGE_THRESHOLD:
                            keep_indices.append(i)
                            large_count += 1

                total_predictions += len(keep_indices)

                # Convert xyxy to xywh and save results
                if keep_indices:
                    keep_indices = torch.tensor(keep_indices)
                    boxes = boxes[keep_indices]
                    scores = scores[keep_indices]
                    labels = labels[keep_indices]

                    # Convert to COCO format
                    boxes[:, 2:] -= boxes[:, :2]  # xyxy to xywh

                    for box, score, label in zip(boxes, scores, labels):
                        results.append({
                            "image_id": img_id,
                            "category_id": int(label.item()),
                            "bbox": [round(x, 2) for x in box.tolist()],
                            "score": float(score.item()),
                        })

    print(f"   Total predictions: {total_predictions}")

    if not results:
        print("⚠️ No predictions above confidence thresholds")
        return 0.0, 0.0, {}

    # COCO evaluation
    try:
        coco_gt.dataset.setdefault("info", {})
        coco_dt = coco_gt.loadRes(results)
        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

        map_50_95 = coco_eval.stats[0]  # mAP@0.5:0.95
        map_50 = coco_eval.stats[1]     # mAP@0.5

        return map_50_95, map_50, results

    except Exception as e:
        print(f"⚠️ Evaluation error: {e}")
        return 0.0, 0.0, {}

In [ ]:
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm   # (ติดตั้ง tqdm ไปแล้ว)


# เพิ่ม cell ใหม่แทนที่ training loop เดิม
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os

# สร้างโฟลเดอร์สำหรับ checkpoints
checkpoint_dir = f"{ROOT}/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Setup ใหม่สำหรับ optimizer และ scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# ใช้ ReduceLROnPlateau แทน StepLR
lr_scheduler = ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3,
    verbose=True, min_lr=1e-6, threshold=0.001
)

# ตัวแปรสำหรับ tracking
num_epochs = 25
loss_hist, map_hist, map50_hist, lr_hist = [], [], [], []
best_map = 0.0
best_map50 = 0.0
patience_counter = 0
early_stop_patience = 7

print("🚀 Starting improved training with enhanced features...")
print(f"   Epochs: {num_epochs}")
print(f"   Early stopping patience: {early_stop_patience}")
print(f"   Checkpoint directory: {checkpoint_dir}")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    epoch_start_time = time.time()

    # Training phase
    pbar = tqdm(train_loader_enhanced, desc=f"E{epoch+1:2d}/{num_epochs}")

    for batch_idx, (images, targets) in enumerate(pbar):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        running_loss += losses.item()

        # Update progress bar
        pbar.set_postfix({
            'Loss': f'{losses.item():.4f}',
            'Avg': f'{running_loss/(batch_idx+1):.4f}',
            'LR': f'{optimizer.param_groups[0]["lr"]:.1e}'
        })

        # Memory cleanup
        if (batch_idx + 1) % 50 == 0:
            torch.cuda.empty_cache()

    # Calculate metrics
    avg_loss = running_loss / len(train_loader_enhanced)
    epoch_time = time.time() - epoch_start_time

    # Evaluation phase
    print(f"\n[Epoch {epoch+1}] Evaluating... ", end="")
    eval_start = time.time()

    """map5095, map50, eval_results = enhanced_evaluate_map(
        model, val_loader_enhanced, device, coco_gt, category_map
    )"""
    map5095, map50, eval_results = size_aware_evaluate_map(
    model, val_loader_enhanced, device, coco_gt, category_map
)


    eval_time = time.time() - eval_start

    # บันทึกประวัติ (แก้ไขปัญหาจากโค้ดเดิม)
    loss_hist.append(avg_loss)
    map_hist.append(map5095)
    map50_hist.append(map50)
    lr_hist.append(optimizer.param_groups[0]['lr'])

    # Update learning rate (แก้ไขลำดับจากโค้ดเดิม)
    lr_scheduler.step(map5095)

    # Display results
    print(f"\n[Epoch {epoch+1:2d}] "
          f"Loss: {avg_loss:.4f} | "
          f"mAP@0.5:0.95: {map5095:.4f} | "
          f"mAP@0.5: {map50:.4f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.1e}")
    print(f"            "
          f"Train time: {epoch_time:.1f}s | "
          f"Eval time: {eval_time:.1f}s")

    # Model checkpointing
    is_best_map = map5095 > best_map
    is_best_map50 = map50 > best_map50

    if is_best_map:
        best_map = map5095
        patience_counter = 0

        # Save best model
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': lr_scheduler.state_dict(),
            'best_map': best_map,
            'best_map50': best_map50,
            'loss_hist': loss_hist,
            'map_hist': map_hist,
            'map50_hist': map50_hist,
            'lr_hist': lr_hist,
            'category_map': category_map,
            'num_classes': num_classes
        }

        torch.save(checkpoint, f'{checkpoint_dir}/best_model_map.pth')
        print(f"✅ New best mAP@0.5:0.95 model saved! ({best_map:.4f})")
    else:
        patience_counter += 1

    if is_best_map50:
        best_map50 = map50
        torch.save(checkpoint, f'{checkpoint_dir}/best_model_map50.pth')
        print(f"✅ New best mAP@0.5 model saved! ({best_map50:.4f})")

    # Analyze detections every 5 epochs
    if (epoch + 1) % 5 == 0 and eval_results:
        analyze_class_performance(eval_results, category_map)

    # Regular checkpoint
    if (epoch + 1) % 5 == 0:
        torch.save(checkpoint, f'{checkpoint_dir}/checkpoint_epoch_{epoch+1}.pth')
        print(f"💾 Regular checkpoint saved (epoch {epoch+1})")

    # Early stopping
    if patience_counter >= early_stop_patience:
        print(f"\n🛑 Early stopping triggered after {epoch+1} epochs")
        print(f"   No improvement in mAP@0.5:0.95 for {early_stop_patience} epochs")
        break

    print("-" * 70)

print(f"\n🎉 Training completed!")
print(f"   Best mAP@0.5:0.95: {best_map:.4f}")
print(f"   Best mAP@0.5: {best_map50:.4f}")
print(f"   Total epochs: {len(loss_hist)}")

# Clear memory
torch.cuda.empty_cache()


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


🚀 Starting improved training with enhanced features...
   Epochs: 25
   Early stopping patience: 7
   Checkpoint directory: /content/drive/MyDrive/Project Deeplearning/TACO/checkpoints


E 1/25: 100%|██████████| 525/525 [05:52<00:00,  1.49it/s, Loss=40.4007, Avg=49.5981, LR=5.0e-03]



[Epoch 1] Evaluating... 🔍 Running size-aware evaluation
   Small objects threshold: 0.3
   Medium objects threshold: 0.4
   Large objects threshold: 0.5


   Total predictions: 0
⚠️ No predictions above confidence thresholds

[Epoch  1] Loss: 49.5981 | mAP@0.5:0.95: 0.0000 | mAP@0.5: 0.0000 | LR: 5.0e-03
            Train time: 352.3s | Eval time: 134.8s
----------------------------------------------------------------------


...

   Total predictions: 10
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.12s).
Accumulating evaluation results...
DONE (t=0.15s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 |

E 4/25: 100%|██████████| 525/525 [03:06<00:00,  2.81it/s, Loss=72.6059, Avg=45.1840, LR=5.0e-03]



[Epoch 4] Evaluating... 🔍 Running size-aware evaluation
   Small objects threshold: 0.3
   Medium objects threshold: 0.4
   Large objects threshold: 0.5


   Total predictions: 0
⚠️ No predictions above confidence thresholds

[Epoch  4] Loss: 45.1840 | mAP@0.5:0.95: 0.0000 | mAP@0.5: 0.0000 | LR: 5.0e-03
            Train time: 186.6s | Eval time: 35.7s
----------------------------------------------------------------------


E 5/25:  77%|███████▋  | 403/525 [02:22<00:36,  3.35it/s, Loss=43.0999, Avg=45.9056, LR=5.0e-03]

In [ ]:
# =============================================================================
# 🧪 COMPREHENSIVE MODEL TESTING SUITE
# =============================================================================

import os
import json
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

# =============================================================================
# 1. TEST DATASET SETUP
# =============================================================================

print("🔄 Setting up test dataset...")

# Load test dataset (assuming you have test_annotations.json)
try:
    with open(f"{ROOT}/test_annotations.json") as f:
        test_json = json.load(f)

    test_dataset = TacoDataset(
        root=f"{ROOT}/test",
        annotations=test_json,
        transforms=EnhancedTransforms(train=False)
    )

    test_loader = DataLoader(
        test_dataset, batch_size=1, shuffle=False,
        collate_fn=collate_fn, num_workers=2, pin_memory=True
    )

    print(f"✅ Test dataset loaded: {len(test_dataset)} samples")

except FileNotFoundError:
    print("⚠️ test_annotations.json not found, using validation set for testing")
    test_dataset = val_dataset_enhanced
    test_loader = val_loader_enhanced

# =============================================================================
# 2. LOAD BEST TRAINED MODEL
# =============================================================================

print("\n🔄 Loading best trained model...")

# Load the best model checkpoint
checkpoint_path = f"{ROOT}/checkpoints/best_model_map.pth"

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])

    print(f"✅ Best model loaded!")
    print(f"   Training mAP@0.5:0.95: {checkpoint.get('best_map', 'N/A'):.4f}")
    print(f"   Training mAP@0.5: {checkpoint.get('best_map50', 'N/A'):.4f}")
    print(f"   Trained for: {checkpoint.get('epoch', 'N/A')} epochs")
else:
    print("⚠️ No checkpoint found, using current model state")
    checkpoint = {}

# Create results directory
results_dir = f"{ROOT}/test_results_{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(results_dir, exist_ok=True)
print(f"📁 Results will be saved to: {results_dir}")

# =============================================================================
# 3. ENHANCED VISUALIZATION FUNCTION
# =============================================================================

def enhanced_visualize_prediction(image, target, prediction, index, save_path=None):
    """Enhanced visualization with side-by-side comparison"""

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    img = image.permute(1, 2, 0).cpu().numpy()

    # === LEFT: Ground Truth ===
    ax1.imshow(img)
    ax1.set_title(f"🎯 Ground Truth - Test Image {index+1}",
                  fontsize=16, fontweight='bold', color='blue')

    gt_count = 0
    gt_classes = set()

    for box, label in zip(target["boxes"].cpu(), target["labels"].cpu()):
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                           fill=False, edgecolor='blue', linewidth=3)
        ax1.add_patch(rect)

        label_name = category_map.get(label.item(), f'unknown_{label.item()}')
        gt_classes.add(label_name)

        # Better text positioning
        text_y = max(y1 - 10, 10)
        ax1.text(x1, text_y, label_name, color='blue', fontsize=11,
                fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3", facecolor='lightblue', alpha=0.9))
        gt_count += 1

    # Ground truth info box
    info_text = f'GT Objects: {gt_count}\nClasses: {len(gt_classes)}'
    ax1.text(0.02, 0.98, info_text, transform=ax1.transAxes,
            verticalalignment='top', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.5", facecolor='lightblue', alpha=0.8))
    ax1.axis('off')

    # === RIGHT: Predictions ===
    ax2.imshow(img)
    ax2.set_title(f"🔍 Model Predictions - Test Image {index+1}",
                  fontsize=16, fontweight='bold', color='red')

    pred_count = 0
    high_conf_count = 0
    pred_classes = set()

    # Sort predictions by confidence
    scores = prediction["scores"].cpu()
    sorted_indices = torch.argsort(scores, descending=True)

    for i in sorted_indices:
        box = prediction["boxes"][i].cpu()
        label = prediction["labels"][i].cpu()
        score = prediction["scores"][i].cpu()

        if score > 0.3:  # Show predictions above 30% confidence
            # Color coding by confidence
            if score > 0.7:
                color, linewidth = 'red', 3
            elif score > 0.5:
                color, linewidth = 'orange', 2.5
            else:
                color, linewidth = 'yellow', 2

            x1, y1, x2, y2 = box
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor=color, linewidth=linewidth)
            ax2.add_patch(rect)

            label_name = category_map.get(label.item(), f'unknown_{label.item()}')
            pred_classes.add(label_name)

            # Better text with confidence
            text_y = max(y1 - 15, 15)
            confidence_text = f"{label_name}\n{score:.3f}"
            ax2.text(x1, text_y, confidence_text, color=color, fontsize=10,
                    fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.9))

            pred_count += 1
            if score > 0.5:
                high_conf_count += 1

    # Predictions info box
    info_text = f'Total Preds: {pred_count}\nHigh Conf (>0.5): {high_conf_count}\nClasses: {len(pred_classes)}'
    ax2.text(0.02, 0.98, info_text, transform=ax2.transAxes,
            verticalalignment='top', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.5", facecolor='lightcoral', alpha=0.8))

    # Add confidence legend
    legend_elements = [
        plt.Rectangle((0,0),1,1, facecolor='red', alpha=0.7, label='High (>0.7)'),
        plt.Rectangle((0,0),1,1, facecolor='orange', alpha=0.7, label='Medium (0.5-0.7)'),
        plt.Rectangle((0,0),1,1, facecolor='yellow', alpha=0.7, label='Low (0.3-0.5)')
    ]
    ax2.legend(handles=legend_elements, loc='upper right', fontsize=10)
    ax2.axis('off')

    plt.tight_layout()

    # Save visualization
    if save_path:
        save_file = f'{save_path}/test_visualization_{index+1:03d}.png'
        plt.savefig(save_file, dpi=300, bbox_inches='tight')
        print(f"💾 Saved: {save_file}")

    plt.show()

# =============================================================================
# 4. COMPREHENSIVE PERFORMANCE EVALUATION
# =============================================================================

def comprehensive_test_evaluation(model, test_loader, device, category_map, save_dir=None):
    """Complete performance evaluation with detailed metrics"""

    model.eval()

    # Storage for results
    all_predictions = []
    all_targets = []
    inference_times = []
    confidence_thresholds = [0.3, 0.5, 0.7, 0.9]
    results_by_threshold = {th: [] for th in confidence_thresholds}

    print(f"\n🔍 Running comprehensive evaluation on {len(test_loader)} images...")

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(tqdm(test_loader, desc="Testing")):
            images = [img.to(device) for img in images]

            # Measure inference time
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            start_time = time.time()

            outputs = model(images)

            torch.cuda.synchronize() if torch.cuda.is_available() else None
            inference_time = time.time() - start_time
            inference_times.append(inference_time)

            # Store results
            for target, output in zip(targets, outputs):
                all_targets.append(target)
                all_predictions.append(output)

                # Analyze different confidence thresholds
                for threshold in confidence_thresholds:
                    valid_preds = output["scores"] > threshold
                    filtered_output = {
                        "boxes": output["boxes"][valid_preds],
                        "labels": output["labels"][valid_preds],
                        "scores": output["scores"][valid_preds]
                    }
                    results_by_threshold[threshold].append({
                        "target": target,
                        "prediction": filtered_output,
                        "image_id": target["image_id"].item() if "image_id" in target else batch_idx
                    })

    # ===== PERFORMANCE ANALYSIS =====

    # Basic Performance Stats
    avg_inference_time = np.mean(inference_times)
    std_inference_time = np.std(inference_times)
    fps = 1.0 / avg_inference_time

    print(f"\n{'='*60}")
    print(f"📊 **PERFORMANCE METRICS**")
    print(f"{'='*60}")
    print(f"Average Inference Time: {avg_inference_time:.4f}s ± {std_inference_time:.4f}s")
    print(f"Frames Per Second (FPS): {fps:.2f}")
    print(f"Min/Max Inference Time: {min(inference_times):.4f}s / {max(inference_times):.4f}s")
    print(f"Total Test Images: {len(all_predictions)}")

    # Confidence Threshold Analysis
    print(f"\n{'='*60}")
    print(f"🎯 **CONFIDENCE THRESHOLD ANALYSIS**")
    print(f"{'='*60}")
    print(f"{'Threshold':<12} {'Total Det':<10} {'Avg/Image':<10} {'Images w/ Det':<15}")
    print(f"{'-'*50}")

    for threshold in confidence_thresholds:
        total_detections = sum(len(r["prediction"]["boxes"]) for r in results_by_threshold[threshold])
        avg_detections = total_detections / len(results_by_threshold[threshold]) if results_by_threshold[threshold] else 0
        images_with_detections = sum(1 for r in results_by_threshold[threshold] if len(r["prediction"]["boxes"]) > 0)

        print(f"{threshold:<12.1f} {total_detections:<10d} {avg_detections:<10.2f} {images_with_detections:<15d}")

    # Class-wise Detection Analysis
    class_detections = defaultdict(int)
    class_confidence_sum = defaultdict(float)
    class_confidence_count = defaultdict(int)

    for pred in all_predictions:
        valid_preds = pred["scores"] > 0.5
        valid_boxes = pred["boxes"][valid_preds]
        valid_labels = pred["labels"][valid_preds]
        valid_scores = pred["scores"][valid_preds]

        for label, score in zip(valid_labels, valid_scores):
            class_name = category_map.get(label.item(), f"unknown_{label.item()}")
            class_detections[class_name] += 1
            class_confidence_sum[class_name] += score.item()
            class_confidence_count[class_name] += 1

    print(f"\n{'='*60}")
    print(f"🏷️ **TOP 15 DETECTED CLASSES (Confidence > 0.5)**")
    print(f"{'='*60}")
    print(f"{'Rank':<4} {'Class Name':<25} {'Count':<8} {'Avg Conf':<10}")
    print(f"{'-'*50}")

    sorted_classes = sorted(class_detections.items(), key=lambda x: x[1], reverse=True)

    for i, (class_name, count) in enumerate(sorted_classes[:15]):
        avg_conf = class_confidence_sum[class_name] / class_confidence_count[class_name]
        print(f"{i+1:<4d} {class_name:<25} {count:<8d} {avg_conf:<10.3f}")

    # Detection Distribution Analysis
    detection_counts = [len(pred["boxes"][pred["scores"] > 0.5]) for pred in all_predictions]

    print(f"\n{'='*60}")
    print(f"📈 **DETECTION DISTRIBUTION ANALYSIS**")
    print(f"{'='*60}")
    print(f"Images with 0 detections: {sum(1 for x in detection_counts if x == 0)}")
    print(f"Images with 1-5 detections: {sum(1 for x in detection_counts if 1 <= x <= 5)}")
    print(f"Images with 6-10 detections: {sum(1 for x in detection_counts if 6 <= x <= 10)}")
    print(f"Images with >10 detections: {sum(1 for x in detection_counts if x > 10)}")
    print(f"Max detections in single image: {max(detection_counts) if detection_counts else 0}")
    print(f"Average detections per image: {np.mean(detection_counts):.2f}")

    return {
        "predictions": all_predictions,
        "targets": all_targets,
        "inference_times": inference_times,
        "class_detections": dict(class_detections),
        "results_by_threshold": results_by_threshold,
        "performance_stats": {
            "avg_inference_time": avg_inference_time,
            "fps": fps,
            "total_images": len(all_predictions),
            "total_detections": sum(class_detections.values())
        }
    }

# =============================================================================
# 5. RUN COMPREHENSIVE TESTING
# =============================================================================

print(f"\n🚀 Starting comprehensive model testing...")
print(f"   Model: {checkpoint_path}")
print(f"   Test images: {len(test_dataset)}")
print(f"   Results directory: {results_dir}")

# Run evaluation
test_results = comprehensive_test_evaluation(
    model, test_loader, device, category_map, results_dir
)

# =============================================================================
# 6. VISUALIZE RANDOM TEST SAMPLES
# =============================================================================

print(f"\n{'='*60}")
print(f"🖼️ **VISUALIZING RANDOM TEST SAMPLES**")
print(f"{'='*60}")

# Set number of visualizations
num_visualizations = min(8, len(test_dataset))
random.seed(42)  # For reproducible selection
random_indices = random.sample(range(len(test_dataset)), num_visualizations)

print(f"Generating {num_visualizations} random visualizations...")

for i, idx in enumerate(random_indices):
    print(f"\nProcessing visualization {i+1}/{num_visualizations} (Image Index: {idx})")

    image, target = test_dataset[idx]
    image = image.to(device)
    target = {k: v.to(device) if torch.is_tensor(v) else v for k, v in target.items()}

    with torch.no_grad():
        prediction = model([image])[0]

    enhanced_visualize_prediction(image, target, prediction, i, results_dir)

# =============================================================================
# 7. GENERATE COMPREHENSIVE TEST REPORT
# =============================================================================

def generate_test_report(test_results, checkpoint, save_path):
    """Generate comprehensive test report in Markdown format"""

    timestamp = time.strftime('%Y-%m-%d %H:%M:%S')
    stats = test_results['performance_stats']

    report = f"""# 🧪 Faster R-CNN Test Report

**Generated:** {timestamp}
**Tester:** Artty02
**Model Checkpoint:** {checkpoint_path}
**Best Training mAP@0.5:0.95:** {checkpoint.get('best_map', 'N/A'):.4f}
**Best Training mAP@0.5:** {checkpoint.get('best_map50', 'N/A'):.4f}
**Training Epochs:** {checkpoint.get('epoch', 'N/A')}

---

## 📊 Performance Summary

| Metric | Value |
|--------|--------|
| **Average Inference Time** | {stats['avg_inference_time']:.4f}s |
| **Frames Per Second (FPS)** | {stats['fps']:.2f} |
| **Total Test Images** | {stats['total_images']} |
| **Total Detections (>0.5 conf)** | {stats['total_detections']} |
| **Average Detections/Image** | {stats['total_detections']/stats['total_images']:.2f} |

---

## 🎯 Confidence Threshold Analysis

| Threshold | Total Detections | Avg per Image |
|-----------|------------------|---------------|"""

    for threshold in [0.3, 0.5, 0.7, 0.9]:
        total_dets = sum(len(r["prediction"]["boxes"]) for r in test_results['results_by_threshold'][threshold])
        avg_dets = total_dets / len(test_results['results_by_threshold'][threshold]) if test_results['results_by_threshold'][threshold] else 0
        report += f"\n| {threshold:.1f} | {total_dets} | {avg_dets:.2f} |"

    report += f"""

---

## 🏆 Top 10 Detected Classes

| Rank | Class Name | Detection Count |
|------|------------|-----------------|"""

    sorted_classes = sorted(test_results['class_detections'].items(),
                          key=lambda x: x[1], reverse=True)

    for i, (class_name, count) in enumerate(sorted_classes[:10]):
        report += f"\n| {i+1} | {class_name} | {count} |"

    report += f"""

---

## 📈 Model Performance Analysis

### Strengths:
- **Inference Speed:** {stats['fps']:.1f} FPS suitable for {'real-time' if stats['fps'] > 15 else 'near real-time'} applications
- **Detection Coverage:** Model detected {len(test_results['class_detections'])} different classes
- **Consistency:** {'Good' if np.std(test_results['inference_times']) < 0.1 else 'Variable'} inference time consistency

### Areas for Improvement:
- Consider confidence threshold optimization
- Monitor class imbalance in detections
- Evaluate false positive rates

---

## 🔧 Technical Details

- **Architecture:** Faster R-CNN ResNet50 FPN V2
- **Dataset:** TACO (Trash Annotations in Context)
- **Training Framework:** PyTorch + Torchvision
- **Evaluation Metrics:** COCO-style mAP
- **Test Environment:** {'GPU' if torch.cuda.is_available() else 'CPU'}

---

*Report generated automatically by the comprehensive testing suite.*
"""

    # Save report
    report_path = f"{save_path}/test_report.md"
    with open(report_path, "w", encoding='utf-8') as f:
        f.write(report)

    print(f"\n📝 **COMPREHENSIVE TEST REPORT GENERATED**")
    print(f"   Report saved: {report_path}")

    return report_path

# Generate the final report
report_path = generate_test_report(test_results, checkpoint, results_dir)

# =============================================================================
# 8. FINAL SUMMARY
# =============================================================================

print(f"\n{'='*60}")
print(f"🎉 **TESTING COMPLETED SUCCESSFULLY!**")
print(f"{'='*60}")
print(f"📁 All results saved to: {results_dir}")
print(f"📊 Performance: {test_results['performance_stats']['fps']:.2f} FPS")
print(f"🎯 Total detections: {test_results['performance_stats']['total_detections']}")
print(f"📝 Full report: {report_path}")
print(f"🖼️ Visualizations: {num_visualizations} samples saved")

# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"🧹 GPU memory cleared")

print(f"\n✅ Ready for deployment or further analysis!")

In [ ]:
# Cell ใหม่ที่ 4: Enhanced Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import time

def plot_enhanced_training_progress(loss_hist, map_hist, map50_hist, lr_hist):
    """Enhanced training progress visualization"""

    plt.style.use('default')
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    epochs = range(1, len(loss_hist) + 1)

    # Loss plot with trend
    axes[0, 0].plot(epochs, loss_hist, 'b-', marker='o', linewidth=2, markersize=4)
    axes[0, 0].set_title('📉 Training Loss Progress', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Average Loss')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].text(0.02, 0.98, f'Final: {loss_hist[-1]:.4f}',
                    transform=axes[0, 0].transAxes, verticalalignment='top',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='lightblue'))

    # mAP comparison
    axes[0, 1].plot(epochs, map_hist, 'g-', marker='s', linewidth=2,
                   markersize=4, label='mAP@0.5:0.95')
    axes[0, 1].plot(epochs, map50_hist, 'orange', marker='^', linewidth=2,
                   markersize=4, label='mAP@0.5')
    axes[0, 1].set_title('📈 Validation mAP Progress', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('mAP Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].text(0.02, 0.98, f'Best mAP@0.5:0.95: {max(map_hist):.4f}\nBest mAP@0.5: {max(map50_hist):.4f}',
                    transform=axes[0, 1].transAxes, verticalalignment='top',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='lightgreen'))

    # Learning rate schedule
    axes[1, 0].plot(epochs, lr_hist, 'r-', marker='^', linewidth=2, markersize=4)
    axes[1, 0].set_title('⚙️ Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Learning Rate')
    axes[1, 0].set_yscale('log')
    axes[1, 0].grid(True, alpha=0.3)

    # Combined metrics
    ax2 = axes[1, 1]
    ax3 = ax2.twinx()

    line1 = ax2.plot(epochs, loss_hist, 'b-', marker='o', linewidth=2,
                    markersize=3, label='Loss', alpha=0.8)
    line2 = ax3.plot(epochs, map_hist, 'g-', marker='s', linewidth=2,
                    markersize=3, label='mAP@0.5:0.95', alpha=0.8)

    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss', color='blue')
    ax3.set_ylabel('mAP@0.5:0.95', color='green')
    ax2.set_title('🔄 Loss vs mAP Correlation', fontsize=14, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor='blue')
    ax3.tick_params(axis='y', labelcolor='green')

    # Legend
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='center right')

    plt.tight_layout()

    # Save plot
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    plt.savefig(f'{ROOT}/training_progress_{timestamp}.png', dpi=300, bbox_inches='tight')
    print(f"📊 Training progress plot saved: training_progress_{timestamp}.png")
    plt.show()

# เรียกใช้ฟังก์ชันนี้หลังจากการฝึกเสร็จ
if len(loss_hist) > 0:  # ตรวจสอบว่ามีข้อมูลแล้ว
    plot_enhanced_training_progress(loss_hist, map_hist, map50_hist, lr_hist)
else:
    print("⚠️ No training history to plot yet. Run training first!")